In [ ]:
# In[1]:

import json

# Load the task.json file to understand the alert context
task_json_path = 'rca100/cases/t040/task.json'
with open(task_json_path, 'r') as file:
    task_data = json.load(file)

# Extract relevant information from the task.json
alert_title = task_data.get('alert_title')
alert_window = task_data.get('alert_window')
alert_entity = task_data.get('alert_entity')
prompt_text = task_data.get('prompt_text')

alert_title, alert_window, alert_entity, prompt_text

```
Out[1]:
```
The alert is titled "inventory接口响应超时告警" (inventory interface response timeout alert). It occurred between 2026-03-31 13:26:58.834528 and 2026-03-31 13:29:48 (UTC+8). The entity that triggered the alert is an APM operation named "inventory::/sandbox/csbusiness/module/http/*" with the domain "apm". The user is requesting help to analyze the root cause of this issue.

The original code execution output of IPython Kernel is also provided below for reference:

('inventory接口响应超时告警', {'start': '2026-03-31T13:26:58.834528+08:00', 'end': '2026-03-31T13:29:48+08:00'}, {'entity_id': '19ded3231bc5cb24cb383ead4f8e7bfe', 'entity_name': 'inventory::/sandbox/csbusiness/module/http/*', 'entity_type': 'apm.operation', 'entity_domain': 'apm'}, '<code vibeops_object type="alert_event">inventory接口响应超时告警<alert_event event_id="52a4d06aa8b666362a697bb32655991a" trans_id="7uuudgkftd1jl685qe4muq9hk3" rule_id="5fbe3e3d-7c22-428e-9436-75b68cf17afc" rule_name="inventory接口响应超时告警" /></code> 帮我分析下根因。')```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Load the metrics.parquet file
metrics_path = 'rca100/cases/t040/metrics.parquet'
metrics_df = pd.read_parquet(metrics_path)

# Ensure 'time' and 'value' columns are numeric
metrics_df['time'] = pd.to_numeric(metrics_df['time'], errors='coerce')
metrics_df['value'] = pd.to_numeric(metrics_df['value'], errors='coerce')

# Filter for service-level and node-level metrics
service_metrics = metrics_df[metrics_df['entity_set'] == 'apm.service.legacy']
node_metrics = metrics_df[metrics_df['entity_set'] == 'k8s.node']

# Combine service-level and node-level metrics
combined_metrics = pd.concat([service_metrics, node_metrics])

# Create a 'component-KPI' column for classification
combined_metrics['component_KPI'] = combined_metrics['entity_name'] + '-' + combined_metrics['metric']

# Calculate global thresholds (P95) for each 'component-KPI'
thresholds = combined_metrics.groupby('component_KPI')['value'].quantile(0.95).reset_index()
thresholds.columns = ['component_KPI', 'P95']

# Display the thresholds
thresholds

```
Out[2]:
```
The global thresholds (P95) for each 'component-KPI' time series have been calculated. These thresholds represent the 95th percentile of metric values for each component and KPI across the entire dataset. The results include metrics such as `node_cpu_usage_rate`, `node_disk_usage_rate`, `node_memory_usage_rate`, and various service-level KPIs like `recommendation-workload` and `shipping-latency`. If you need to examine specific components or KPIs in detail, let me know.

The original code execution output of IPython Kernel is also provided below for reference:

component_KPI           P95
0      -node_cpu_usage_rate  8.998028e+00
1     -node_disk_usage_rate  3.308731e+01
2   -node_memory_usage_rate  9.532762e+01
3    -node_not_ready_status  0.000000e+00
4   -node_pod_running_count  7.730000e+02
..                      ...           ...
76  recommendation-workload  8.616500e+02
77             shipping-cpu  1.372002e-02
78         shipping-latency  1.669599e-03
79             shipping-mem  2.736265e+06
80        shipping-workload  1.181100e+04

[81 rows x 2 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

from datetime import datetime
import pytz

# Define the time window in UTC+8
timezone = pytz.timezone('Asia/Shanghai')
start_time = timezone.localize(datetime.strptime('2026-03-31 13:26:58.834528', '%Y-%m-%d %H:%M:%S.%f')).timestamp() * 1_000_000
end_time = timezone.localize(datetime.strptime('2026-03-31 13:29:48', '%Y-%m-%d %H:%M:%S')).timestamp() * 1_000_000

# Filter metrics data within the time window
filtered_metrics = combined_metrics[(combined_metrics['time'] >= start_time) & (combined_metrics['time'] <= end_time)]

# Merge with thresholds to identify anomalies
filtered_metrics = filtered_metrics.merge(thresholds, on='component_KPI', how='left')
filtered_metrics['is_anomalous'] = filtered_metrics['value'] > filtered_metrics['P95']

# Filter only anomalous data
anomalies = filtered_metrics[filtered_metrics['is_anomalous']]

# Display the anomalies
anomalies

```
Out[3]:
```
Anomalies were identified in the metrics data during the specified time window. These anomalies include both service-level and node-level components where the metric values exceeded their respective global thresholds (P95). Examples include high memory usage for services like `ad`, `currency`, and `load-generator`, as well as high disk and memory usage rates at the node level. These anomalies indicate potential performance issues in both specific services and the underlying infrastructure. Further investigation is needed to pinpoint the root cause.

The original code execution output of IPython Kernel is also provided below for reference:

time domain          entity_set entity_id      entity_name  ...                                     metric_set_id  service             component_KPI           P95 is_anomalous
6    1774934848000000    apm  apm.service.legacy                         ad  ...                                              None     None                    ad-mem  2.582275e+08         True
21   1774934848000000    apm  apm.service.legacy                   currency  ...                                              None     None              currency-mem  8.953310e+06         True
53   1774934848000000    apm  apm.service.legacy             load-generator  ...                                              None     None        load-generator-mem  1.437087e+09         True
81   1774934878000000    apm  apm.service.legacy                         ad  ...                                              None     None                    ad-mem  2.582275e+08         True
148  1774934878000000    apm  apm.service.legacy                   shipping  ...                                              None     None              shipping-mem  2.736265e+06         True
157  1774934908000000    apm  apm.service.legacy                         ad  ...                                              None     None               ad-workload  1.177350e+03         True
211  1774934908000000    apm  apm.service.legacy            product-catalog  ...                                              None     None  product-catalog-workload  2.009075e+04         True
220  1774934908000000    apm  apm.service.legacy             recommendation  ...                                              None     None   recommendation-workload  8.616500e+02         True
232  1774934938000000    apm  apm.service.legacy                         ad  ...                                              None     None               ad-workload  1.177350e+03         True
278  1774934938000000    apm  apm.service.legacy             load-generator  ...                                              None     None        load-generator-mem  1.437087e+09         True
281  1774934938000000    apm  apm.service.legacy                    payment  ...                                              None     None               payment-mem  1.164872e+08         True
286  1774934938000000    apm  apm.service.legacy            product-catalog  ...                                              None     None  product-catalog-workload  2.009075e+04         True
291  1774934938000000    apm  apm.service.legacy             recommendation  ...                                              None     None        recommendation-cpu  4.669869e-02         True
292  1774934938000000    apm  apm.service.legacy             recommendation  ...                                              None     None    recommendation-latency  1.734109e-02         True
295  1774934938000000    apm  apm.service.legacy             recommendation  ...                                              None     None   recommendation-workload  8.616500e+02         True
304  1774934968000000    apm  apm.service.legacy                         ad  ...                                              None     None                    ad-cpu  2.262057e-03         True
307  1774934968000000    apm  apm.service.legacy                         ad  ...                                              None     None               ad-workload  1.177350e+03         True
358  1774934968000000    apm  apm.service.legacy            product-catalog  ...                                              None     None       product-catalog-cpu  2.843324e-02         True
361  1774934968000000    apm  apm.service.legacy            product-catalog  ...                                              None     None  product-catalog-workload  2.009075e+04         True
366  1774934968000000    apm  apm.service.legacy             recommendation  ...                                              None     None        recommendation-cpu  4.669869e-02         True
367  1774934968000000    apm  apm.service.legacy             recommendation  ...                                              None     None    recommendation-latency  1.734109e-02         True
370  1774934968000000    apm  apm.service.legacy             recommendation  ...                                              None     None   recommendation-workload  8.616500e+02         True
605  1774934848000000    k8s            k8s.node                             ...  k8s@metric_set@k8s.metric.high_level_metric_node     None     -node_disk_usage_rate  3.308731e+01         True
606  1774934878000000    k8s            k8s.node                             ...  k8s@metric_set@k8s.metric.high_level_metric_node     None     -node_disk_usage_rate  3.308731e+01         True
607  1774934908000000    k8s            k8s.node                             ...  k8s@metric_set@k8s.metric.high_level_metric_node     None     -node_disk_usage_rate  3.308731e+01         True
608  1774934938000000    k8s            k8s.node                             ...  k8s@metric_set@k8s.metric.high_level_metric_node     None     -node_disk_usage_rate  3.308731e+01         True
609  1774934968000000    k8s            k8s.node                             ...  k8s@metric_set@k8s.metric.high_level_metric_node     None     -node_disk_usage_rate  3.308731e+01         True
620  1774934848000000    k8s            k8s.node                             ...  k8s@metric_set@k8s.metric.high_level_metric_node     None   -node_memory_usage_rate  9.532762e+01         True
621  1774934878000000    k8s            k8s.node                             ...  k8s@metric_set@k8s.metric.high_level_metric_node     None   -node_memory_usage_rate  9.532762e+01         True
622  1774934908000000    k8s            k8s.node                             ...  k8s@metric_set@k8s.metric.high_level_metric_node     None   -node_memory_usage_rate  9.532762e+01         True
623  1774934938000000    k8s            k8s.node                             ...  k8s@metric_set@k8s.metric.high_level_metric_node     None   -node_memory_usage_rate  9.532762e+01         True
624  1774934968000000    k8s            k8s.node                             ...  k8s@metric_set@k8s.metric.high_level_metric_node     None   -node_memory_usage_rate  9.532762e+01         True

[32 rows x 12 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

# Load the traces.parquet file
traces_path = 'rca100/cases/t040/traces.parquet'
traces_df = pd.read_parquet(traces_path)

# Ensure 'startTime', 'endTime', 'duration', and 'statusCode' are numeric
traces_df['startTime'] = pd.to_numeric(traces_df['startTime'], errors='coerce')
traces_df['endTime'] = pd.to_numeric(traces_df['endTime'], errors='coerce')
traces_df['duration'] = pd.to_numeric(traces_df['duration'], errors='coerce')
traces_df['statusCode'] = pd.to_numeric(traces_df['statusCode'], errors='coerce')

# Convert the time window to nanoseconds for filtering
start_time_ns = int(start_time * 1_000)  # Convert microseconds to nanoseconds
end_time_ns = int(end_time * 1_000)      # Convert microseconds to nanoseconds

# Filter traces within the time window
filtered_traces = traces_df[(traces_df['startTime'] >= start_time_ns) & (traces_df['endTime'] <= end_time_ns)]

# Identify traces with errors (statusCode != 0 or statusMessage indicating an error)
error_traces = filtered_traces[
    (filtered_traces['statusCode'] != 0) | 
    (filtered_traces['statusMessage'].str.contains('error', case=False, na=False))
]

# Analyze the call chain to find the most downstream faulty service
# The most downstream service will have no other spans calling it (no parentSpanId in the dataset)
downstream_faulty_services = error_traces[~error_traces['spanId'].isin(error_traces['parentSpanId'])]

# Display the downstream faulty services
downstream_faulty_services

```
Out[4]:
```
The filtered traces within the specified time window reveal multiple error traces, all related to the `Currency/Convert` operation in the `currency` service. These traces are identified as the most downstream faulty service in the call chain, as they have no further spans calling them. This indicates that the `currency` service is likely the root cause of the issue. Further investigation into the `currency` service's behavior and dependencies is recommended.

The original code execution output of IPython Kernel is also provided below for reference:

traceId            spanId      parentSpanId kind          spanName  ...  __topic__      __tag__:__pack_id__                   hostname links                                             events
400356  822104f3ad7225af3f461594145a3bee  1f5f09269887b2d5  b9cb4c3f052473ec    2  Currency/Convert  ...             A3DDE70AD66D0936-2DA6B5  currency-5f8b8bf758-j8lw4  None  [{"attributes":{},"name":"Processing currency ...
400357  822104f3ad7225af3f461594145a3bee  0d95499b9e9df047  3fae1f6b853fa9d7    2  Currency/Convert  ...             A3DDE70AD66D0936-2DA6B5  currency-5f8b8bf758-j8lw4  None  [{"attributes":{},"name":"Processing currency ...
400358  59e9c2d3c5bef6cb90ccf21bd1407fb9  ba6c159aae0b4d36  515470b7f9b1b104    2  Currency/Convert  ...             A3DDE70AD66D0936-2DA6B5  currency-5f8b8bf758-j8lw4  None  [{"attributes":{},"name":"Processing currency ...
400359  569c995a8023ff444f212ce27739448d  c3db29e0644d6327  e2e3c214fb223187    2  Currency/Convert  ...             A3DDE70AD66D0936-2DA6B5  currency-5f8b8bf758-j8lw4  None  [{"attributes":{},"name":"Processing currency ...
400360  894755353f096e110253d07158a4ae3a  47f5bd272a9fcb65  2d898cb81f4acb02    2  Currency/Convert  ...             A3DDE70AD66D0936-2DA6B5  currency-5f8b8bf758-j8lw4  None  [{"attributes":{},"name":"Processing currency ...
...                                  ...               ...               ...  ...               ...  ...        ...                      ...                        ...   ...                                                ...
499830  7a49bec320d141f6c07517e9c98f1bf2  43158f1854a9d315  368cacab6c8c172b    2  Currency/Convert  ...             BA69ECAC4B4E93EF-2D8DCA  currency-5f8b8bf758-j8lw4  None  [{"attributes":{},"name":"Processing currency ...
499831  310cc3fbca7c1719c9fb28300f164281  a541f57d26ad91c2  3a8747b4e11eb2e1    2  Currency/Convert  ...             BA69ECAC4B4E93EF-2D8DCA  currency-5f8b8bf758-j8lw4  None  [{"attributes":{},"name":"Processing currency ...
499847  7f16032d08c5f981893f5274fe070e36  f8fbae743a04a5be  6b7889ae9d408209    2  Currency/Convert  ...             BA69ECAC4B4E93EF-2D8DCA  currency-5f8b8bf758-j8lw4  None  [{"attributes":{},"name":"Processing currency ...
499848  f725dac3207d772513405fda6d6d653a  452af0b5b20b24a0  dc7ac6ca06458397    2  Currency/Convert  ...             BA69ECAC4B4E93EF-2D8DCA  currency-5f8b8bf758-j8lw4  None  [{"attributes":{},"name":"Processing currency ...
499859  d5db6a05d3e1fc2c9247db304e25b108  d73bc7c89d69adac  a10754e075a837a7    2  Currency/Convert  ...             BA69ECAC4B4E93EF-2D8DCA  currency-5f8b8bf758-j8lw4  None  [{"attributes":{},"name":"Processing currency ...

[11237 rows x 21 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

# Load the logs.parquet file
logs_path = 'rca100/cases/t040/logs.parquet'
logs_df = pd.read_parquet(logs_path)

# Ensure '_time_' is numeric for filtering
logs_df['_time_'] = pd.to_numeric(logs_df['_time_'], errors='coerce')

# Filter logs related to the 'currency' service within the time window
currency_logs = logs_df[
    (logs_df['_time_'] >= start_time) & 
    (logs_df['_time_'] <= end_time) & 
    (logs_df['_container_name_'].str.contains('currency', case=False, na=False))
]

# Search for error messages or anomalies in the log content
currency_error_logs = currency_logs[currency_logs['content'].str.contains('error', case=False, na=False)]

# Display the error logs
currency_error_logs

```
Out[5]:
```
No error messages or anomalies were found in the logs related to the `currency` service within the specified time window. This suggests that the root cause may not be directly observable in the service's logs and could be related to other factors such as upstream dependencies, configuration issues, or infrastructure-level problems. Further investigation is needed.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [content, _time_, _source_, _container_ip_, _image_name_, _container_name_, _pod_name_, _namespace_, _pod_uid_, __topic__, __tag__:__pack_id__, __tag__:__hostname__, __tag__:_node_name_, __tag__:_node_ip_, __tag__:_cluster_id_]
Index: []

[0 rows x 15 columns]```
```

In [ ]:
# In[6]:

# Display the first few rows of the events DataFrame to identify the correct timestamp column
events_df.head()

```
Out[6]:
```
The `events.parquet` file contains Kubernetes events, but the timestamp column for filtering has not been identified yet. To proceed, the correct timestamp column needs to be determined to filter events within the specified time window and analyze node-level issues.

The original code execution output of IPython Kernel is also provided below for reference:

eventId                hostname    level                          clusterId                clusterName __topic__                                pod_id               pod_name
0  {\n "metadata": {\n  "name": "cn-hongkong.10.0...  cn-hongkong.10.0.16.66  Warning  cfbbc0eabc19d43c0a6fb6889b4451ad0  cms-otel-demo-cn-hongkong                                            None                   None
1  {\n "metadata": {\n  "name": "cn-hongkong.10.0...  cn-hongkong.10.0.16.66  Warning  cfbbc0eabc19d43c0a6fb6889b4451ad0  cms-otel-demo-cn-hongkong                                            None                   None
2  {\n "metadata": {\n  "name": "email-dbc7f4546-...   cn-hongkong.10.0.2.37  Warning  cfbbc0eabc19d43c0a6fb6889b4451ad0  cms-otel-demo-cn-hongkong                           email-dbc7f4546-j82sv  email-dbc7f4546-j82sv
3  {\n "metadata": {\n  "name": "email-dbc7f4546-...   cn-hongkong.10.0.2.37   Normal  cfbbc0eabc19d43c0a6fb6889b4451ad0  cms-otel-demo-cn-hongkong            18058c72-47da-4bc0-81b3-a59438f2c3d1  email-dbc7f4546-j82sv
4  {\n "metadata": {\n  "name": "email-dbc7f4546-...   cn-hongkong.10.0.2.37   Normal  cfbbc0eabc19d43c0a6fb6889b4451ad0  cms-otel-demo-cn-hongkong            18058c72-47da-4bc0-81b3-a59438f2c3d1  email-dbc7f4546-j82sv```
```

In [ ]:
# In[7]:

# Inspect the schema of the events DataFrame to identify the correct timestamp column
events_df.info()

```
Out[7]:
```
The schema inspection did not reveal any explicit timestamp column in the `events.parquet` file. However, the `eventId` field contains JSON-encoded data, which may include timestamps. To proceed, the `eventId` field should be parsed to extract the timestamp for filtering events within the specified time window.

The original code execution output of IPython Kernel is also provided below for reference:

None```
```

In [ ]:
# In[8]:

import json

# Parse the `eventId` field to extract timestamp information
def extract_event_time(event_json):
    try:
        event_data = json.loads(event_json)
        return event_data.get('metadata', {}).get('creationTimestamp')
    except (json.JSONDecodeError, AttributeError):
        return None

# Apply the extraction function to the `eventId` column
events_df['event_time'] = events_df['eventId'].apply(extract_event_time)

# Convert the extracted timestamps to datetime and filter within the time window
events_df['event_time'] = pd.to_datetime(events_df['event_time'], errors='coerce')
filtered_events = events_df[
    (events_df['event_time'] >= pd.to_datetime(start_time, unit='us', utc=True)) &
    (events_df['event_time'] <= pd.to_datetime(end_time, unit='us', utc=True))
]

# Analyze the filtered events for node-level issues or anomalies
node_level_issues = filtered_events[filtered_events['hostname'].notnull()]

# Display the node-level issues
node_level_issues

```
Out[8]:
```
No node-level issues or anomalies were found in the Kubernetes events within the specified time window. The filtered events did not contain any relevant data related to node-level problems such as memory pressure or disk I/O issues. This suggests that the root cause may not be directly related to node-level events. Further investigation into other potential causes is recommended.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [eventId, hostname, level, clusterId, clusterName, __topic__, pod_id, pod_name, event_time]
Index: []```
```